[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jivatneet/strategic-feature-selection/blob/main/upcoding_data_generation_copy.ipynb)

In [6]:
%%writefile upcoding_v28_top10_ha2025_export.R
# ---------------------------
# Export a V28 HCC upcoding benchmark
# ---------------------------
# Output structure:
#   OUT_ROOT/
#     baseline_raw.csv
#     baseline_hier.csv
#     baseline_feature_meta.csv
#     target_mapping_table.csv
#     scenario_manifest.csv
#     scenario_low/
#       scenario_spec.csv
#       all_hcc_upcoded_data_pre_hierarchy.csv
#       all_hcc_upcoded_data.csv
#       feature_meta.csv
#       target_audit.csv
#       shift_summary.csv
#       event_summary.csv
#       calibration_log.csv
#     scenario_main/
#       ...
#     scenario_high/
#       ...

library(upcoding)
library(data.table)
library(readr)
library(dplyr)
library(tibble)
library(purrr)
library(stringr)

# ---------------------------
# User-adjustable parameters
# ---------------------------
OUT_ROOT <- "upcoding_v28_top10_ha2025_export"
NUM_PEOPLE <- 5000L
BASELINE_SEED <- 123L
MANIPULATION_SEED <- 456L

NUM_TIMEPOINTS <- 1L
CENSORING_PROP <- 0.0

APPLY_HIERARCHY_BASELINE <- TRUE
APPLY_HIERARCHY_AFTER_UPCODING <- TRUE

UNDERCODING_PROP <- 0.0

SCENARIO_TBL <- tibble(
  scenario = c("scenario_low", "scenario_main", "scenario_high"),
  shift_scale = c(0.50, 0.75, 1.00)
)

HEART_FAILURE_TARGET <- "hcc225"       # Alternatives: "hcc224", "hcc226"
HEART_FAILURE_APPROACH <- "any"        # Alternatives: "lower severity"
CKD_TARGET <- "hcc328"                 # Alternatives: "hcc329"
CKD_APPROACH <- "any"                  # Alternatives: "lower severity"

# Binary-search calibration settings for each target under a single-target run.
CALIBRATION_ITERS <- 12L
CALIBRATION_TOL <- 2e-4

# The upcoding package requires upcoding_prop to lie strictly between 0 and 1.
# We therefore use 0 to mean "skip this target" in our own code and never pass
# 0 or 1 to upcode_all_hccs().
MIN_UPCODING_PROP <- 1e-8
MAX_UPCODING_PROP <- 0.999

# ---------------------------
# Helpers
# ---------------------------
`%||%` <- function(x, y) if (is.null(x)) y else x

fresh_read_dt <- function(path) {
  dt <- fread(path)
  if ("person_id" %in% names(dt)) {
    # The upcoding package samples row IDs using person_id but writes by row index.
    # Resetting person_id to the row order makes this invariant explicit.
    dt[, person_id := seq_len(.N)]
  }
  dt
}

get_hcc_cols <- function(dt) {
  setdiff(names(dt), "person_id")
}

apply_hierarchy_if_needed <- function(dt) {
  apply_v28_cmshcc_hierarchy(copy(dt))
}

write_feature_meta <- function(baseline_dt, manipulated_dt = NULL, scenario_name = "baseline", out_path) {
  hcc_cols <- get_hcc_cols(baseline_dt)
  base_prev <- vapply(hcc_cols, function(h) mean(baseline_dt[[h]]), numeric(1))
  if (is.null(manipulated_dt)) {
    meta <- tibble(
      feature = hcc_cols,
      baseline_prevalence = base_prev,
      manipulated_prevalence = NA_real_,
      prevalence_shift = NA_real_,
      scenario = scenario_name
    )
  } else {
    manip_prev <- vapply(hcc_cols, function(h) mean(manipulated_dt[[h]]), numeric(1))
    meta <- tibble(
      feature = hcc_cols,
      baseline_prevalence = base_prev,
      manipulated_prevalence = manip_prev,
      prevalence_shift = manip_prev - base_prev,
      scenario = scenario_name
    )
  }
  write_csv(meta, out_path)
  invisible(meta)
}

row_sum_summary <- function(dt, label) {
  hcc_cols <- get_hcc_cols(dt)
  rs <- rowSums(as.matrix(dt[, ..hcc_cols]))
  tibble(
    dataset = label,
    n = nrow(dt),
    d = length(hcc_cols),
    mean_active_hccs = mean(rs),
    median_active_hccs = median(rs),
    p75_active_hccs = quantile(rs, 0.75),
    max_active_hccs = max(rs),
    zero_row_fraction = mean(rs == 0),
    nonzero_hcc_columns = sum(colMeans(as.matrix(dt[, ..hcc_cols])) > 0)
  )
}

summarize_shift <- function(baseline_dt, manipulated_dt) {
  hcc_cols <- get_hcc_cols(baseline_dt)
  base_mat <- as.matrix(baseline_dt[, ..hcc_cols])
  manip_mat <- as.matrix(manipulated_dt[, ..hcc_cols])
  row_changes <- rowSums(base_mat != manip_mat)
  tibble(
    frac_rows_changed = mean(row_changes > 0),
    mean_num_feature_changes = mean(row_changes),
    median_num_feature_changes = median(row_changes),
    max_num_feature_changes = max(row_changes)
  )
}

calibrate_single_target <- function(baseline_dt, target_hcc, approach, desired_prev, tmp_parent, seed_base) {
  hcc_cols <- get_hcc_cols(baseline_dt)
  base_prev <- mean(baseline_dt[[target_hcc]])

  if (desired_prev <= base_prev + CALIBRATION_TOL) {
    return(list(prop = 0.0, achieved = base_prev, base_prev = base_prev,
                desired_prev = desired_prev, feasible = TRUE, note = "already_at_or_above_target"))
  }

  eval_prop <- function(prop, iter_id) {
    # Do not pass boundary values to the package validator.
    prop <- min(max(prop, MIN_UPCODING_PROP), MAX_UPCODING_PROP)
    tmp_dir <- file.path(tmp_parent, paste0("calib_", target_hcc, "_", iter_id, "_", sprintf("%.6f", prop)))
    dir.create(tmp_dir, recursive = TRUE, showWarnings = FALSE)
    spec <- tibble(hcc = target_hcc, approach = approach, upcoding_prop = prop)
    upcode_all_hccs(
      copy(baseline_dt),
      spec,
      out_dir = tmp_dir,
      num_timepoints = NUM_TIMEPOINTS,
      censoring_prop = CENSORING_PROP,
      curr_seed = seed_base + iter_id
    )
    mpath <- file.path(tmp_dir, "all_hcc_upcoded_data.csv")
    mdt <- fresh_read_dt(mpath)
    if (APPLY_HIERARCHY_AFTER_UPCODING) {
      mdt <- apply_hierarchy_if_needed(mdt)
    }
    achieved <- mean(mdt[[target_hcc]])
    unlink(tmp_dir, recursive = TRUE, force = TRUE)
    achieved
  }

  lo <- 0.0
  hi <- MAX_UPCODING_PROP
  best_prop <- hi
  best_achieved <- eval_prop(hi, 999L)
  feasible <- best_achieved >= desired_prev - CALIBRATION_TOL

  if (!feasible) {
    return(list(prop = MAX_UPCODING_PROP, achieved = best_achieved, base_prev = base_prev,
                desired_prev = desired_prev, feasible = FALSE, note = "source_pool_limited_at_cap"))
  }

  for (it in seq_len(CALIBRATION_ITERS)) {
    mid <- (lo + hi) / 2
    achieved <- eval_prop(mid, it)
    if (achieved >= desired_prev) {
      best_prop <- mid
      best_achieved <- achieved
      hi <- mid
    } else {
      lo <- mid
    }
    if (abs(achieved - desired_prev) <= CALIBRATION_TOL) {
      best_prop <- mid
      best_achieved <- achieved
      break
    }
  }
  list(prop = best_prop, achieved = best_achieved, base_prev = base_prev,
       desired_prev = desired_prev, feasible = TRUE, note = "calibrated")
}

# ---------------------------
# Target table: V28 top-10 groups from Kronick et al. 2025
# ---------------------------
# ma_prev and tm_prev are group-level measured prevalences from Exhibit 2.
# For grouped HCCs, target_hcc is the representative upcoding target used in the simulator.
# approach = "lower severity" uses the package's competing-event hierarchy;
# approach = "any" upcodes any eligible row not already coded with the target.
TARGET_TABLE <- tibble(
  target_hcc = c(
    "hcc280",  # COPD / chronic lung disorders
    HEART_FAILURE_TARGET,  # acute or chronic heart failure, except end-stage
    "hcc36",   # diabetes severe/acute complications: severity-based from lower diabetes HCCs
    "hcc48",   # morbid obesity
    "hcc137",  # drug use disorder
    CKD_TARGET,  # CKD stage 3 / moderate CKD
    "hcc155",  # major depression, moderate/severe, without psychosis
    "hcc93",   # rheumatoid arthritis and other specified inflammatory rheumatic disorders
    "hcc253",  # hemiplegia/hemiparesis
    "hcc139"   # alcohol use disorder
  ),
  approach = c(
    "any",
    HEART_FAILURE_APPROACH,
    "lower severity",
    "any",
    "any",
    CKD_APPROACH,
    "any",
    "any",
    "any",
    "any"
  ),
  diagnosis_group = c(
    "COPD, interstitial lung disorders, and other chronic lung disorders",
    "Acute or chronic heart failure, except end-stage",
    "Diabetes with acute or chronic complications",
    "Morbid obesity",
    "Drug use disorder, moderate/severe, or non-psychotic complications",
    "Chronic kidney disease, moderate, stage 3",
    "Major depression, moderate/severe, without psychosis",
    "Rheumatoid arthritis and other specified inflammatory rheumatic disorders",
    "Hemiplegia/hemiparesis",
    "Alcohol use disorder"
  ),
  health_affairs_group_hccs = c(
    "HCC280",
    "HCC224, HCC225, HCC226",
    "HCC36, HCC37, HCC38",
    "HCC48",
    "HCC137",
    "HCC328, HCC329",
    "HCC155",
    "HCC93",
    "HCC253",
    "HCC139"
  ),
  ma_prev = c(0.17, 0.13, 0.31, 0.12, 0.04, 0.14, 0.05, 0.04, 0.02, 0.02),
  tm_prev = c(0.11, 0.10, 0.24, 0.06, 0.02, 0.09, 0.03, 0.03, 0.01, 0.01),
  v28_risk_coefficient = c(0.314, 0.374, 0.175, 0.192, 0.425, 0.134, 0.278, 0.565, 0.385, 0.255),
  removal_effect_percent = c(1.8, 1.3, 1.2, 1.1, 0.9, 0.6, 0.5, 0.5, 0.2, 0.2),
  rationale = c(
    "V28 top-10 group; any-available upcoding is plausible for chronic lung diagnoses.",
    paste0("V28 top-10 group. Default any-available target is ", toupper(HEART_FAILURE_TARGET), "; severity-based heart-failure upcoding was source-pool limited in the sparse baseline."),
    "V28 top-10 group; severity-based upcoding from lower diabetes HCCs is clinically natural.",
    "V28 top-10 group; any-available upcoding.",
    "V28 top-10 group; any-available upcoding because lower-severity source pools may be sparse.",
    paste0("V28 top-10 group. Default any-available target is ", toupper(CKD_TARGET), "; severity-based CKD upcoding was source-pool limited in the sparse baseline."),
    "V28 top-10 group; HCC154 is not included in the new V28 top-10 table, so only HCC155 is targeted.",
    "V28 top-10 group; any-available upcoding.",
    "V28 top-10 group; any-available upcoding.",
    "V28 top-10 group; any-available upcoding."
  )
) %>%
  mutate(prevalence_lift = pmax(ma_prev - tm_prev, 0))

# ---------------------------
# Main export
# ---------------------------
dir.create(OUT_ROOT, showWarnings = FALSE, recursive = TRUE)
write_csv(TARGET_TABLE, file.path(OUT_ROOT, "target_mapping_table.csv"))

# Generate baseline from the raw All-of-Us co-occurrence table, not the dense imputed default.
simulate_baseline_v28_hcc_dt(
  num_people = NUM_PEOPLE,
  reference_cooccurring_hcc_count = cooccurring_v28_hcc_counts_allofus,
  apply_v28_hierarchy = APPLY_HIERARCHY_BASELINE,
  out_dir = OUT_ROOT,
  out_file_prefix = "baseline_raw",
  curr_seed = BASELINE_SEED
)

baseline_raw <- fresh_read_dt(file.path(OUT_ROOT, "baseline_raw.csv"))
if (APPLY_HIERARCHY_BASELINE) {
  baseline_hier <- apply_hierarchy_if_needed(baseline_raw)
} else {
  baseline_hier <- copy(baseline_raw)
}
baseline_hier[, person_id := seq_len(.N)]
fwrite(baseline_hier, file.path(OUT_ROOT, "baseline_hier.csv"))

baseline_for_upcoding <- copy(baseline_hier)
if (UNDERCODING_PROP > 0) {
  baseline_for_upcoding <- undercode_dt(copy(baseline_for_upcoding), undercoding_prop = UNDERCODING_PROP)
  baseline_for_upcoding[, person_id := seq_len(.N)]
}
fwrite(baseline_for_upcoding, file.path(OUT_ROOT, "baseline_tm_like.csv"))
write_feature_meta(baseline_for_upcoding, NULL, "baseline", file.path(OUT_ROOT, "baseline_feature_meta.csv"))
write_csv(row_sum_summary(baseline_for_upcoding, "baseline"), file.path(OUT_ROOT, "baseline_row_sum_summary.csv"))

manifest_rows <- list()

for (rr in seq_len(nrow(SCENARIO_TBL))) {
  scenario_name <- SCENARIO_TBL$scenario[rr]
  shift_scale <- SCENARIO_TBL$shift_scale[rr]
  scenario_dir <- file.path(OUT_ROOT, scenario_name)
  dir.create(scenario_dir, showWarnings = FALSE, recursive = TRUE)

  tmp_parent <- file.path(scenario_dir, "_calibration_tmp")
  dir.create(tmp_parent, showWarnings = FALSE, recursive = TRUE)

  calib_rows <- list()
  spec_rows <- list()
  for (ii in seq_len(nrow(TARGET_TABLE))) {
    row <- TARGET_TABLE[ii, ]
    target <- row$target_hcc
    desired_prev <- mean(baseline_for_upcoding[[target]]) + shift_scale * row$prevalence_lift
    desired_prev <- min(desired_prev, MAX_UPCODING_PROP)
    calib <- calibrate_single_target(
      baseline_dt = baseline_for_upcoding,
      target_hcc = target,
      approach = row$approach,
      desired_prev = desired_prev,
      tmp_parent = tmp_parent,
      seed_base = MANIPULATION_SEED + 1000L * rr + 100L * ii
    )
    calib_rows[[ii]] <- tibble(
      scenario = scenario_name,
      shift_scale = shift_scale,
      target_hcc = target,
      approach = row$approach,
      base_prev = calib$base_prev,
      desired_prev = calib$desired_prev,
      single_target_achieved_prev = calib$achieved,
      calibrated_upcoding_prop = calib$prop,
      feasible_single_target = calib$feasible,
      calibration_note = calib$note,
      diagnosis_group = row$diagnosis_group
    )
    spec_rows[[ii]] <- tibble(
      hcc = target,
      approach = row$approach,
      upcoding_prop = calib$prop
    )
  }
  unlink(tmp_parent, recursive = TRUE, force = TRUE)

  calibration_log <- bind_rows(calib_rows)
  spec_df_full <- bind_rows(spec_rows) %>%
    mutate(
      # Keep the full table for audit, but never pass boundary values to the package.
      upcoding_prop = pmin(pmax(upcoding_prop, 0.0), MAX_UPCODING_PROP),
      pass_to_upcoding = upcoding_prop > MIN_UPCODING_PROP
    )
  spec_df <- spec_df_full %>%
    filter(pass_to_upcoding) %>%
    select(hcc, approach, upcoding_prop)

  write_csv(calibration_log, file.path(scenario_dir, "calibration_log.csv"))
  write_csv(spec_df_full, file.path(scenario_dir, "scenario_spec_full_audit.csv"))
  write_csv(spec_df, file.path(scenario_dir, "scenario_spec.csv"))

  if (nrow(spec_df) > 0) {
    upcode_all_hccs(
      copy(baseline_for_upcoding),
      spec_df,
      out_dir = scenario_dir,
      num_timepoints = NUM_TIMEPOINTS,
      censoring_prop = CENSORING_PROP,
      curr_seed = MANIPULATION_SEED + rr
    )
    manipulated_pre <- fresh_read_dt(file.path(scenario_dir, "all_hcc_upcoded_data.csv"))
  } else {
    manipulated_pre <- copy(baseline_for_upcoding)
  }

  fwrite(manipulated_pre, file.path(scenario_dir, "all_hcc_upcoded_data_pre_hierarchy.csv"))

  if (APPLY_HIERARCHY_AFTER_UPCODING) {
    manipulated_final <- apply_hierarchy_if_needed(manipulated_pre)
  } else {
    manipulated_final <- copy(manipulated_pre)
  }
  manipulated_final[, person_id := seq_len(.N)]
  fwrite(manipulated_final, file.path(scenario_dir, "all_hcc_upcoded_data.csv"))

  write_feature_meta(baseline_for_upcoding, manipulated_final, scenario_name, file.path(scenario_dir, "feature_meta.csv"))
  write_csv(summarize_shift(baseline_for_upcoding, manipulated_final), file.path(scenario_dir, "shift_summary.csv"))
  write_csv(row_sum_summary(manipulated_final, scenario_name), file.path(scenario_dir, "row_sum_summary.csv"))

  target_audit <- TARGET_TABLE %>%
    transmute(
      scenario = scenario_name,
      shift_scale = shift_scale,
      target_hcc,
      approach,
      diagnosis_group,
      health_affairs_group_hccs,
      ma_prev,
      tm_prev,
      prevalence_lift,
      desired_prev = pmin(sapply(target_hcc, function(h) mean(baseline_for_upcoding[[h]])) + shift_scale * prevalence_lift, MAX_UPCODING_PROP),
      baseline_prev = sapply(target_hcc, function(h) mean(baseline_for_upcoding[[h]])),
      final_prev = sapply(target_hcc, function(h) mean(manipulated_final[[h]])),
      final_shift = final_prev - baseline_prev,
      abs_target_error = abs(final_prev - desired_prev)
    )
  write_csv(target_audit, file.path(scenario_dir, "target_audit.csv"))

  event_files <- list.files(scenario_dir, pattern = "_upcoded_data_event_and_time_labels\\.csv$", full.names = TRUE)
  if (length(event_files) > 0) {
    event_summary <- map_dfr(event_files, function(fp) {
      ev <- fread(fp)
      tibble(
        file = basename(fp),
        rows = nrow(ev),
        event_columns = paste(names(ev), collapse = "|")
      )
    })
  } else {
    event_summary <- tibble(file = character(), rows = integer(), event_columns = character())
  }
  write_csv(event_summary, file.path(scenario_dir, "event_summary.csv"))

  manifest_rows[[rr]] <- target_audit %>%
    mutate(num_people = NUM_PEOPLE,
           num_timepoints = NUM_TIMEPOINTS,
           censoring_prop = CENSORING_PROP,
           undercoding_prop = UNDERCODING_PROP,
           apply_hierarchy_baseline = APPLY_HIERARCHY_BASELINE,
           apply_hierarchy_after_upcoding = APPLY_HIERARCHY_AFTER_UPCODING)
}

scenario_manifest <- bind_rows(manifest_rows)
write_csv(scenario_manifest, file.path(OUT_ROOT, "scenario_manifest.csv"))

message("Done. Export folder: ", normalizePath(OUT_ROOT))

Writing upcoding_v28_top10_ha2025_export.R


In [2]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq r-base r-base-dev
!Rscript -e "install.packages(c('pak','data.table','readr','dplyr','tibble','purrr','stringr'), repos='https://cloud.r-project.org')"
!Rscript -e "if (!requireNamespace('upcoding', quietly=TRUE)) pak::pak('StanfordHPDS/upcoding')"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 4.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
(Reading database ... 122807 files and directories currently installed.)
Preparing to unpack .../r-base-core_4.6.1-9.2404.0_amd64.deb ...
Unpacking r-base-core (4.6.1-9.2404.0) over (4.6.1-6.2404.0) ...
Preparing to unpack .../r-base_4.6.1-9.2404.0_all.deb ...
Unpacking r-base (4.6.1-9.2404.0) over (4.6.1-6.2404.0) ...
Preparing to unpack .../r-recommende

In [5]:
!Rscript /content/upcoding_v28_top10_ha2025_export.R


Attaching package: ‘data.table’

The following object is masked from ‘package:base’:

    %notin%


Attaching package: ‘dplyr’

The following objects are masked from ‘package:data.table’:

    between, first, last

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Attaching package: ‘purrr’

The following object is masked from ‘package:data.table’:

    transpose

Done. Export folder: /content/upcoding_v28_top10_ha2025_export


In [7]:
%%writefile cms_v28_cna_coefficients_export_features.csv
feature,hcc_code,description,cna_coef,source_page
hcc1,1,HIV/AIDS,0.301,184
hcc2,2,"Septicemia, Sepsis, Systemic Inflammatory Response Syndrome/Shock",0.5,184
hcc6,6,Opportunistic Infections,0.381,184
hcc17,17,"Cancer Metastatic to Lung, Liver, Brain, and Other Organs; Acute Myeloid Leukemia Except Promyelocytic",4.209,184
hcc18,18,"Cancer Metastatic to Bone, Other and Unspecified Metastatic Cancer; Acute Leukemia Except Myeloid",2.341,184
hcc19,19,"Myelodysplastic Syndromes, Multiple Myeloma, and Other Cancers",1.798,184
hcc20,20,Lung and Other Severe Cancers,1.136,184
hcc21,21,Lymphoma and Other Cancers,0.671,184
hcc22,22,"Bladder, Colorectal, and Other Cancers",0.363,184
hcc23,23,"Prostate, Breast, and Other Cancers and Tumors",0.186,184
hcc35,35,Pancreas Transplant Status,0.949,184
hcc36,36,Diabetes with Severe Acute Complications,0.166,184
hcc37,37,Diabetes with Chronic Complications,0.166,184
hcc38,38,"Diabetes with Glycemic, Unspecified, or No Complications",0.166,184
hcc48,48,Morbid Obesity,0.186,184
hcc49,49,Specified Lysosomal Storage Disorders,9.256,184
hcc50,50,"Amyloidosis, Porphyria, and Other Specified Metabolic Disorders",0.648,185
hcc51,51,"Addison's and Cushing's Diseases, Acromegaly, and Other Specified Endocrine Disorders",0.51,185
hcc62,62,Liver Transplant Status/Complications,0.376,185
hcc63,63,Chronic Liver Failure/End- Stage Liver Disorders,0.962,185
hcc64,64,Cirrhosis of Liver,0.447,185
hcc65,65,Chronic Hepatitis,0.185,185
hcc68,68,Cholangitis and Obstruction of Bile Duct Without Gallstones,0.388,185
hcc77,77,Intestine Transplant Status/Complications,1.172,185
hcc78,78,Intestinal Obstruction/Perforation,0.326,185
hcc79,79,Chronic Pancreatitis,0.357,185
hcc80,80,Crohn's Disease (Regional Enteritis),0.55,185
hcc81,81,Ulcerative Colitis,0.244,185
hcc92,92,Bone/Joint/Muscle/Severe Soft Tissue Infections/Necrosis,0.479,185
hcc93,93,Rheumatoid Arthritis and Other Specified Inflammatory Rheumatic Disorders,0.617,185
hcc94,94,Systemic Lupus Erythematosus and Other Specified Systemic Connective Tissue Disorders,0.268,185
hcc107,107,Sickle Cell Anemia (Hb-SS) and Thalassemia Beta Zero,0.457,185
hcc108,108,"Sickle Cell Disorders, Except Sickle Cell Anemia (Hb-SS) and Thalassemia Beta Zero; Beta Thalassemia Major",0.146,186
hcc109,109,"Acquired Hemolytic, Aplastic, and Sideroblastic Anemias",1.144,186
hcc111,111,"Hemophilia, Male",4.639,186
hcc112,112,Immune Thrombocytopenia and Specified Coagulation Defects and Hemorrhagic Conditions,0.45,186
hcc114,114,Common Variable and Combined Immunodeficiencies,2.262,186
hcc115,115,Specified Immunodeficiencies and White Blood Cell Disorders,0.565,186
hcc125,125,"Dementia, Severe",0.341,186
hcc126,126,"Dementia, Moderate",0.341,186
hcc127,127,"Dementia, Mild or Unspecified",0.341,186
hcc135,135,Drug Use with Psychotic Complications,0.424,186
hcc136,136,Alcohol Use with Psychotic Complications,0.424,186
hcc137,137,"Drug Use Disorder, Moderate/Severe, or Drug Use with Non-Psychotic Complications",0.424,186
hcc138,138,"Drug Use Disorder, Mild, Uncomplicated, Except Cannabis",0.423,186
hcc139,139,"Alcohol Use Disorder, Moderate/Severe, or Alcohol Use with Specified Non- Psychotic Complications",0.242,186
hcc151,151,Schizophrenia,0.511,187
hcc152,152,"Psychosis, Except Schizophrenia",0.484,187
hcc153,153,Personality Disorders; Anorexia/Bulimia Nervosa,0.396,187
hcc154,154,Bipolar Disorders without Psychosis,0.351,187
hcc155,155,"Major Depression, Moderate or Severe, without Psychosis",0.299,187
hcc180,180,Quadriplegia,1.125,187
hcc181,181,Paraplegia,0.942,187
hcc182,182,Spinal Cord Disorders/Injuries,0.478,187
hcc190,190,"Amyotrophic Lateral Sclerosis and Other Motor Neuron Disease, Spinal Muscular Atrophy",1.175,187
hcc191,191,Quadriplegic Cerebral Palsy,0.855,187
hcc192,192,"Cerebral Palsy, Except Quadriplegic",0.314,187
hcc193,193,Chronic Inflammatory Demyelinating Polyneuritis and Multifocal Motor Neuropathy,1.692,187
hcc195,195,Myasthenia Gravis with (Acute) Exacerbation,2.909,187
hcc196,196,Myasthenia Gravis without (Acute) Exacerbation and Other Myoneural Disorders,0.516,187
hcc197,197,Muscular Dystrophy,0.426,187
hcc198,198,Multiple Sclerosis,0.647,187
hcc199,199,Parkinson and Other Degenerative Disease of Basal Ganglia,0.615,187
hcc200,200,Friedreich and Other Hereditary Ataxias; Huntington Disease,0.279,188
hcc201,201,Seizure Disorders and Convulsions,0.245,188
hcc202,202,"Coma, Brain Compression/Anoxic Damage",0.543,188
hcc211,211,Respirator Dependence/Tracheostomy Status/Complications,0.879,188
hcc212,212,Respiratory Arrest,0.37,188
hcc213,213,Cardio-Respiratory Failure and Shock,0.37,188
hcc221,221,Heart Transplant Status/Complications,1.053,188
hcc222,222,End-Stage Heart Failure,2.505,188
hcc223,223,Heart Failure with Heart Assist Device/Artificial Heart,2.505,188
hcc224,224,Acute on Chronic Heart Failure,0.36,188
hcc225,225,Acute Heart Failure (Excludes Acute on Chronic),0.36,188
hcc226,226,"Heart Failure, Except End- Stage and Acute",0.36,188
hcc227,227,Cardiomyopathy/Myocarditis,0.189,188
hcc228,228,Acute Myocardial Infarction,0.252,188
hcc229,229,Unstable Angina and Other Acute Ischemic Heart Disease,0.24,188
hcc238,238,Specified Heart Arrhythmias,0.299,188
hcc248,248,Intracranial Hemorrhage,0.239,188
hcc249,249,Ischemic or Unspecified Stroke,0.239,188
hcc253,253,Hemiplegia/Hemiparesis,0.387,188
hcc254,254,"Monoplegia, Other Paralytic Syndromes",0.321,188
hcc263,263,Atherosclerosis of Arteries of the Extremities with Ulceration or Gangrene,1.118,189
hcc264,264,Vascular Disease with Complications,0.455,189
hcc267,267,Deep Vein Thrombosis and Pulmonary Embolism,0.294,189
hcc276,276,Lung Transplant Status/Complications,2.531,189
hcc277,277,Cystic Fibrosis,0.998,189
hcc278,278,Idiopathic Pulmonary Fibrosis and Lung Involvement in Systemic Sclerosis,0.818,189
hcc279,279,Severe Persistent Asthma,0.818,189
hcc280,280,"Chronic Obstructive Pulmonary Disease, Interstitial Lung Disorders, and Other Chronic Lung Disorders",0.319,189
hcc282,282,Aspiration and Specified Bacterial Pneumonias,0.44,189
hcc283,283,"Empyema, Lung Abscess",0.204,189
hcc298,298,"Severe Diabetic Eye Disease, Retinal Vein Occlusion, and Vitreous Hemorrhage",0.336,189
hcc300,300,Exudative Macular Degeneration,0.596,189
hcc326,326,"Chronic Kidney Disease, Stage 5",0.815,189
hcc327,327,"Chronic Kidney Disease, Severe (Stage 4)",0.514,189
hcc328,328,"Chronic Kidney Disease, Moderate (Stage 3B)",0.127,189
hcc329,329,"Chronic Kidney Disease, Moderate (Stage 3, Except 3B)",0.127,189
hcc379,379,"Pressure Ulcer of Skin with Necrosis Through to Muscle, Tendon, or Bone",1.965,190
hcc380,380,"Chronic Ulcer of Skin, Except Pressure, Through to Bone or Muscle",1.078,190
hcc381,381,Pressure Ulcer of Skin with Full Thickness Skin Loss,1.075,190
hcc382,382,Pressure Ulcer of Skin with Partial Thickness Skin Loss,0.838,190
hcc383,383,"Chronic Ulcer of Skin, Except Pressure, Not Specified as Through to Bone or Muscle",0.646,190
hcc385,385,Severe Skin Burn,1.291,190
hcc387,387,"Pemphigus, Pemphigoid, and Other Specified Autoimmune Skin Disorders",0.406,190
hcc397,397,Major Head Injury with Loss of Consciousness > 1 Hour,0.199,190
hcc398,398,Major Head Injury with Loss of Consciousness < 1 Hour or Unspecified,0.199,190
hcc399,399,Major Head Injury without Loss of Consciousness,0.199,190
hcc401,401,Vertebral Fractures without Spinal Cord Injury,0.522,190
hcc402,402,Hip Fracture/Dislocation,0.467,190
hcc405,405,Traumatic Amputations and Complications,0.598,190
hcc409,409,"Amputation Status, Lower Limb/Amputation Complications",0.598,190
hcc454,454,"Stem Cell, Including Bone Marrow, Transplant Status/Complications",1.068,191
hcc463,463,Artificial Openings for Feeding or Elimination,0.673,191

Writing cms_v28_cna_coefficients_export_features.csv


In [8]:
!zip -r upcoding_v28_top10_ha2025_export.zip /content/upcoding_v28_top10_ha2025_export

  adding: content/upcoding_v28_top10_ha2025_export/ (stored 0%)
  adding: content/upcoding_v28_top10_ha2025_export/scenario_manifest.csv (deflated 81%)
  adding: content/upcoding_v28_top10_ha2025_export/baseline_raw.csv (deflated 97%)
  adding: content/upcoding_v28_top10_ha2025_export/baseline_feature_meta.csv (deflated 84%)
  adding: content/upcoding_v28_top10_ha2025_export/scenario_high/ (stored 0%)
  adding: content/upcoding_v28_top10_ha2025_export/scenario_high/row_sum_summary.csv (deflated 33%)
  adding: content/upcoding_v28_top10_ha2025_export/scenario_high/calibration_log.csv (deflated 58%)
  adding: content/upcoding_v28_top10_ha2025_export/scenario_high/hcc139_upcoded_data_event_and_time_labels.csv (deflated 73%)
  adding: content/upcoding_v28_top10_ha2025_export/scenario_high/shift_summary.csv (deflated 39%)
  adding: content/upcoding_v28_top10_ha2025_export/scenario_high/hcc155_upcoded_data_event_and_time_labels.csv (deflated 73%)
  adding: content/upcoding_v28_top10_ha2025_e